In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, find_peaks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import SGD, Adam


In [6]:
def preprocess_emg_signal(file_path, fs = 1000, mvc_dict=None):
    df = pd.read_csv(file_path)
    df_processed = df.copy()
    channels = [f'channel{i}' for i in range(1, 9)]
    nyq = 0.5 * fs
    b_band, a_band = butter(4, [20/nyq, 450/nyq], btype='band')
    b_low, a_low = butter(4, 10/nyq, btype='low')

    for ch in channels:
        if ch not in df.columns:
            print(f"Warning: cannot find {ch} in the file!")
            continue
        raw_sig = df[ch].fillna(0).values
        filtered = filtfilt(b_band, a_band, raw_sig)
        rectified = np.abs(filtered)
        envelope = filtfilt(b_low, a_low, rectified)
        envelope = np.maximum(envelope, 0)
        if mvc_dict is not None and ch in mvc_dict:
            mvc = mvc_dict[ch]
        else:
            mvc = np.max(envelope)
            if mvc == 0: mvc = 1

        normalized = (envelope / mvc) * 100

        df_processed[f'{ch}_norm'] = normalized

        print(df_processed)

    output_file = file_path.replace('.csv', '_processed.csv')
    df_processed.to_csv(output_file, index=False)

    print("Finishing Preprocessing dataset")

    return df_processed


In [7]:
file_path = r"datasets\train\EMG-data.csv"
df = preprocess_emg_signal(file_path)
df 

          time  channel1  channel2  channel3  channel4  channel5  channel6  \
0            1   0.00001  -0.00002  -0.00001  -0.00003   0.00000  -0.00001   
1            5   0.00001  -0.00002  -0.00001  -0.00003   0.00000  -0.00001   
2            6  -0.00001   0.00001   0.00002   0.00000   0.00001  -0.00002   
3            7  -0.00001   0.00001   0.00002   0.00000   0.00001  -0.00002   
4            8  -0.00001   0.00001   0.00002   0.00000   0.00001  -0.00002   
...        ...       ...       ...       ...       ...       ...       ...   
4237902  50962   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   
4237903  50963   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   
4237904  50964   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   
4237905  50965   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   
4237906  50966   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   

         channel7  channel8  class  label  channel1_norm  
0   

,time,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,class,label,channel1_norm,channel2_norm,channel3_norm,channel4_norm,channel5_norm,channel6_norm,channel7_norm,channel8_norm
0,1,0.00001,-0.00002,-0.00001,-0.00003,0.00000,-0.00001,0.00000,-0.00001,0,1,0.305582,0.346350,0.000000,0.405888,0.000000,0.054135,0.017472,0.000000
1,5,0.00001,-0.00002,-0.00001,-0.00003,0.00000,-0.00001,0.00000,-0.00001,0,1,0.326472,0.423523,0.051696,0.532532,0.000000,0.113909,0.032576,0.000000
2,6,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1,0.347065,0.502241,0.201114,0.658244,0.000000,0.172636,0.048004,0.000000
3,7,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1,0.367312,0.582400,0.349487,0.782807,0.000000,0.230206,0.063736,0.000000
4,8,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1,0.387167,0.663891,0.496581,0.906009,0.000000,0.286517,0.079752,0.004179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237902,50962,0.00001,-0.00001,-0.00002,-0.00004,-0.00012,0.00000,0.00002,0.00002,0,36,1.218362,0.569168,0.896970,0.841954,3.641126,3.038713,1.596788,2.066380
4237903,50963,0.00001,-0.00001,-0.00002,-0.00004,-0.00012,0.00000,0.00002,0.00002,0,36,1.216188,0.569136,0.896311,0.842641,3.644920,3.037181,1.598525,2.066052
4237904,50964,0.00001,-0.00001,-0.00002,-0.00004,-0.00012,0.00000,0.00002,0.00002,0,36,1.214376,0.569113,0.895760,0.843211,3.648063,3.035916,1.599968,2.065764
4237905,50965,0.00001,-0.00001,-0.00002,-0.00004,-0.00012,0.00000,0.00002,0.00002,0,36,1.212884,0.569095,0.895304,0.843679,3.650631,3.034884,1.601151,2.065515
